In [ ]:
import numpy as np
import tomlkit
import pandas as pd
import yaml
from IPython.display import display
import importlib
import copy

import study_lib

In [ ]:
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt

In [ ]:
importlib.reload(study_lib)
do_run = study_lib.do_run 
run_experiment = study_lib.run_experiment
config_series = study_lib.config_series

In [ ]:
base_config_yaml = """
candidates: 5
voters: 13
considerations:
- Likability:
    mean: 0.5
- Irrational:
    sigma: 1.0
    camps: 0
    individualism_deg: 30
- Issues:
    - halfcsep: 0.0
      halfvsep: 0.0
      sigma: 1.0
methods:
- Plurality:
    strat: Honest
- Range:
    nranks: 10
    strat: Honest
- Range:
    nranks: 2
    strat: Honest
- Range:
    nranks: 2
    strat: Strategic
- InstantRunoff: {}
- Borda: {}
- Multivote:
    spread_fact: 1.0
    strat: Honest
    votes: 3
- STAR:
    strat: Honest
- STAR:
    strat: Strategic
    strategic_stretch_factor: 1.5
"""
config = yaml.safe_load(base_config_yaml)
base_config = copy.deepcopy(config)

In [ ]:
config = yaml.safe_load(base_config_yaml)
df = run_experiment(
    config_series(config, 'voters', [7, 9, 11, 13, 15, 51]),
    trials=10000
)
df

More voters makes regrets lower

In [ ]:
config = yaml.safe_load(base_config_yaml)
df = run_experiment(
    config_series(config, 'considerations.0.Likability.mean', [0., 0.5, 1.0, 4.0, 40.0]),
    trials=100000
)
df

In [ ]:
config = yaml.safe_load(base_config_yaml)
df = run_experiment(
    config_series(config, 'considerations.0.Likability.mean', [0., 0.5, 1.0, 4.0, 40.0]),
    trials=100000
)
df

Likability makes voting systems more nearly equivalent.

Not sure why multivoting doesn't do better than this though.

In [ ]:
config['considerations'] = [
    {'Likability': {'mean': 1.0}},
]
df = run_experiment(
    config_series(config, 'considerations.0.Likability.mean', [0.5, 1.0, 4.0, 40.0]),
    trials=100000
)
df

Above is a test of pure likability. This uncovered a serious bug. Likability originally wasn't adding to scores, it was overwriting them. I should design the code better, probably.

It makes sense that plurality, IRV, and Borda all give perfect scores. In this scenario, all voters have the same scores and issue the same ballots. Range, approval, multivoting, and star can all give ties.

In [ ]:
config = yaml.safe_load(base_config_yaml)
df = run_experiment(
    config_series(config, 'considerations.1.Irrational.sigma', [0., 0.5, 2.0, 4.0, 40.0]),
    trials=100000
)
df

Irrational voters make everything tougher.

In [ ]:
config = yaml.safe_load(base_config_yaml)
config['considerations'] = config['considerations'][1:2]  # Irr only
df = run_experiment(
    config_series(config, 'considerations.0.Irrational.camps', [0, 2, 3, 4, 5]),
    trials=100000
)
df

In [ ]:
# Try something more politically-divided. Partly because this gives plurality fits. ;D
config = copy.deepcopy(base_config)
config['considerations'] = yaml.safe_load('''
- Likability:
    mean: 0.1
- Irrational:
    sigma: 0.25
    camps: 0
    individualism_deg: 30
- Issues:
    - halfcsep: 1.5
      halfvsep: 1.5
      sigma: 1.0
''')
df = run_experiment(
    config_series(config, 'considerations.2.Issues.0.sigma', [0, .25, 0.75, 1.0, 1.5, 2, 10]),
    trials=100000
)
df

In [ ]:
# Have to do this as one experiment, since number of votes is part of the column name.
# Also it's more efficient this way.
config = copy.deepcopy(base_config)
config['considerations'] = yaml.safe_load('''
- Likability:
    mean: 0.1
- Irrational:
    sigma: 0.25
    camps: 0
    individualism_deg: 30
- Issues:
    - halfcsep: 1.5
      halfvsep: 1.5
      sigma: 1.0
''')
config['methods'].extend(yaml.safe_load('''
- Multivote:
    spread_fact: 1.0
    strat: Honest
    votes: 2
- Multivote:
    spread_fact: 1.0
    strat: Honest
    votes: 4
- RP:
    strat: Honest
'''))
config['methods'][1]['Range']['nranks'] = 5
df = run_experiment([config], trials=1000000, pi=True)
df

In [ ]:
data = [
    ['Plurality', df.pl_h_mR[0]],
    ['IRV', df.IRV_h_mR[0]],
    ['Approval', df.aprv_h_mR[0]],
    ['Multi, 2', df.multi_h_2v_mR[0]],
    ['Multi, 3', df.multi_h_3v_mR[0]],
    ['Multi, 4', df.multi_h_4v_mR[0]],
    ['Borda', df.Borda_h_mR[0]],
    ['Score 1-5', df.range_5_h_mR[0]],
    # ['STAR', df.star_6_h_mR[0]],
    ['Ranked Pairs', df.rp_h_mR[0]],
]
data_x = [p[0] for p in data]
data_y = [(1.0 - p[1]) * 100. for p in data]

In [ ]:
ax = sns.barplot(x=data_x, y=data_y, errorbar=None, hue=data_x, legend=False)
plt.xlabel('Voting method')
plt.ylabel('Voter Satisfaction (%)')
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
print(ax.containers)
for i in range(len(data_x)):
    ax.containers[i].datavalues = [float(f'{data_y[i]:.1f}')]
    ax.bar_label(ax.containers[i], fmt='{:.1f}')

In [ ]:
data_y

In [ ]:
ax = sns.barplot(x=data_x, y=data_y, errorbar=None, hue=data_x, legend=False)
plt.xlabel('Voting method')
plt.ylabel('Voter Satisfaction (%)')
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
print(ax.containers)
for i in range(len(data_x)):
    ax.containers[i].datavalues = [float(f'{data_y[i]:.1f}')]
    ax.bar_label(ax.containers[i], fmt='{:.1f}')

I think the plot above is the "money plot." For my particular group, I was considering either approval or multivoting with two or three votes.

* Plurality is just there for reference.
* IRV is about equivalent to what was done before, without the "instant" part.
* Approval is my recommendation on the basis of this study.
* With multivoting, it's hard to know how much to spread your votes.
* Multivoting-4, Borda, Score, and Ranked Pairs are also for reference, not seriously considered.

In this small group, strategic voting is not really a concern so Borda is an option, except that this is a small group of friends voting on friends. It is awkward to divulge too much preference in one ballot. So scoring and complete ranking ballots are not options, again just in my own use case at the moment.

Also in my case, extreme simplicity is needed. That eliminates ranked pairs and IRV. Even score voting and Borda are a stretch because the counting is done by hand.

Now I could try partial Borda -- list top N (N=2 probably) only. Also provide an offset, so for N=2, top choice gets 2 pts plus offset, next one gets 1 plus offset.

In [ ]:
data = [
    ['Plurality', df.pl_h_pi[0]],
    ['IRV', df.IRV_h_pi[0]],
    ['Approval', df.aprv_h_pi[0]],
    ['Multivoting, 2', df.multi_h_2v_pi[0]],
    ['Multivoting, 3', df.multi_h_3v_pi[0]],
    ['Borda', df.Borda_h_pi[0]],
    ['Range 1-10', df.range_5_h_pi[0]],
    ['STAR', df.star_6_h_pi[0]],
    ['RP', df.rp_h_pi[0]],
]
data_x = [p[0] for p in data]
data_y = [p[1] for p in data]
ax = sns.barplot(x=data_x, y=data_y, errorbar=None, hue=data_x, legend=False)
plt.xlabel('Voting method')
plt.ylabel('% best candidate elected')
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
print(ax.containers)
for i in range(len(data_x)):
    ax.bar_label(ax.containers[i], fmt='{:.1f}')